In [1]:
!pip install gradio

  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/42.9 MB ? eta -:--:--
   -- ------------------------------------- 2.4/42.9 MB 13.4 MB/s eta 0:00:04
   ----- ---------------------------------- 5.5/42.9 MB 14.6 MB/s eta 0:00:03
   ------- -------------------------------- 8.4/42.9 MB 14.4 MB/s eta 0:00:03
   ---------- ----------------------------- 11.5/42.9 MB 14.1 MB/s eta 0:00:03
   ------------- -------------------------- 14.2/42.9 MB 14.1 MB/s eta 0:00:03
   --------------- ------------------------ 17.0/42.9 MB 13.8 MB/s eta 0:00:02
   ------------------ --------------------- 20.2/42.9 MB 13.9 MB/s eta 0:00:02
   --------------------- ------------------ 22.5/42.9 MB 14.1 MB/s eta 0:00:02
   ---------------------- ----------------- 23.9/42.9 MB 12.8 MB/s eta 0:00:02
   ------------------------ --------------- 26.0/42.9 MB 12.4 MB/s eta 0:00:02
   -------------------------- ------------- 28.8/42.9 MB 12.4 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.21.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.6.0 which is incompatible.
transformers 5.3.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.21.1 which is incompatible.
unstructured-client 0.42.10 requires httpcore>=1.0.9, but you have httpcore 1.0.7 which is incompatible.
unstructured-client 0.42.10 requires pydantic>=2.11.2, but you have pydantic 2.10.6 which is incompatible.


In [2]:
import os
import gradio as gr
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

c:\Users\adira\anaconda3\envs\langchainenvjan17\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [5]:
# 2. Load Nestlé HR Policy
filename = "the_nestle_hr_policy_pdf_2012.pdf"
doc=[]
doc.extend(PyPDFLoader(filename).load())
print(f"Example excerpt:\n", doc[0].page_content[:200])


Example excerpt:
 Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy


In [6]:
# 2. Chunk with metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(doc)
for i, chunk in enumerate(chunks):
    chunk.metadata["source"] = chunk.metadata.get("source", filename[0])  # fallback to any file
    chunk.metadata["chunk_id"] = i + 1
print(f"Total chunks: {len(chunks)}")

Total chunks: 20


In [7]:
# 3. Embedding
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # Or use HuggingFaceEmbeddings


In [8]:
# 4. Vector store (FAISS, local & free)
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding=embeddings)

In [9]:
# 1. Save FAISS vectorstore (after building)
vectorstore.save_local("Nestle_HR_Policy")

In [10]:
# 5. Retrieval chain (LCEL style)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

system_message = SystemMessagePromptTemplate.from_template("""
    "You are a helpful Nestlé HR Assistant. "
    "Use the following pieces of retrieved context to answer the user's question about HR policies. "
    "If the answer is not in the context, say you don't know.\n\n"
    
""")

human_message = HumanMessagePromptTemplate.from_template("""
Use the following context to answer the user's question:

{context}

Question: {input}
Answer:
""")

prompt = ChatPromptTemplate.from_messages([system_message, human_message])
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)

In [11]:
from langchain_core.runnables import RunnableMap, RunnableLambda

qa_chain = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(x["input"])
    })
    | RunnableLambda(lambda inputs: {
        "answer": combine_docs_chain.invoke(inputs),
        "source_documents": inputs["context"]
    })
)


In [12]:
questions = [
    "What is the core mission of the Human Resources function at Nestlé?",
    "Who holds the prime responsibility for sustaining a committed work environment?",
    "What are the three dedicated areas that make up the HR structure?",
    "How does Nestlé support a balance between private and professional life?"
]

for q in questions:
    result = qa_chain.invoke({"input": q})
    print(f"\nQ: {q}\nA: {result['answer']}\n")
    print("--- Source chunk(s) ---")
    for doc in result["source_documents"]:
        print(f"[source: {doc.metadata.get('source')}, chunk {doc.metadata.get('chunk_id')}]")
        print(doc.page_content, "\n")


Q: What is the core mission of the Human Resources function at Nestlé?
A: The core mission of the Human Resources function at Nestlé is to provide professional guidance to line managers aiming to deliver superior business results by optimizing the performance of employees while ensuring exemplary working conditions.

--- Source chunk(s) ---
[source: the_nestle_hr_policy_pdf_2012.pdf, chunk 5]
The Nestlé Human Resources Policy
2
Line managers have the prime responsibility for 
building and sustaining an environment where 
people have a sense of personal commitment 
to their work and give their best to ensure our 
Company’s success. They care for and develop 
the leaders of tomorrow.
Line managers decide on all people matters 
under their influence, within the boundaries set 
by the policies and principles, acting as the final 
decision makers.
The Human Resources (HR) structure 
enables and empowers them in establishing 
business needs and their corresponding people 
requirements. 
The